[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_14/SFM_ch14_seminar/SFM_ch14_seminar.ipynb)

# Statistics of Financial Markets — Seminar 14: Crypto assets

*Seminar notebook. Daniel Traian Pele, Bucharest University of Economic Studies.*

- This seminar comes before Lecture 14: the slides "What You Need for Today" give every definition used here.
- Part A: computations on paper, checked in code. Part B: real data with inference and an interpretation question. Part C: an open question and an AI answer to audit.
- **[Solved]**: complete code and output, a model to follow. **[Proposed]**: write your own code in the empty cell.

> Quantlet `SFM_ch14_seminar`: student version of the Seminar 14 notebook. Solved exercises (A1, A3, A5, A7, B1, B3, B5, B7) are shown with code and output; the proposed exercises (A2, A4, A6, A8, B2, B4, B6, B8, C1, C2) are not solved here (an empty code cell where they need code).

> The solutions are discussed in the seminar.

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_14'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# the arch package (maximum-likelihood estimation of GARCH models) is not part of Google Colab
!pip install -q arch

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

- Transparent background, legend below the plot, course palette.

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

- Daily market data from EODHD, saved in `data/market` of the SFM repository (read locally, or from GitHub in Colab); EUR/RON is the official BNR reference rate.
- Returns in %: $r_t = 100(\ln P_t - \ln P_{t-1})$; each series on its own calendar.

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Seminar functions

- Returns and joint returns; moments and Hill tail indices; ACF and Ljung–Box; GARCH(1,1)-t; VaR and ES by historical simulation and Student-t; the Kupiec test; peg statistics; a rolling VaR backtest; crisis episodes.
- Helpers for Part A and Part B: annualisation, drawdown of a price path, peg deviations in bp, a Normal crypto position, the worst returns of a series, the Hill plot, the Fisher $z$ test, VaR of a position.

In [ ]:
import matplotlib.dates as mdates
import statsmodels.api as sm
from arch import arch_model
EXTRA = {'usdc': ('USDC-USD.CC', 'USD Coin (USDC)', 'Crypto', 'close', '2018-10-09'), 'dai': ('DAI-USD.CC', 'Dai (DAI)', 'Crypto', 'close', '2017-12-27'), 'ibit': ('IBIT.US', 'iShares Bitcoin Trust (IBIT)', 'ETF', 'adjusted_close', '2024-01-11')}
NAME = {'btc': 'Bitcoin', 'eth': 'Ethereum', 'sol': 'Solana', 'sp500': 'S&P 500', 'gold': 'Gold', 'usdt': 'Tether (USDT)', 'usdc': 'USD Coin (USDC)', 'dai': 'Dai (DAI)', 'ibit': 'IBIT'}
START = {'btc': '2014-09-17', 'eth': '2016-01-01', 'sol': '2020-04-11', 'sp500': '2014-09-17', 'gold': '2014-09-17'}
ASSETS = ['btc', 'eth', 'sol', 'sp500', 'gold']
COLORS = {'btc': '#E67E22', 'eth': '#8E44AD', 'sol': '#17A2B8', 'sp500': '#1A3A6E', 'gold': '#B5853F', 'usdt': '#2E7D32', 'usdc': '#1A3A6E', 'dai': '#E67E22', 'total': '#CD0000', 'ibit': '#1A3A6E'}
HALVINGS = ['2012-11-28', '2016-07-09', '2020-05-11', '2024-04-20']
ETF_DATE = '2024-01-11'
EVENTS = {'COVID-19': '2020-03-12', 'Terra/Luna': '2022-05-09', 'FTX': '2022-11-08', 'spot ETFs': '2024-01-11'}
EPISODES = {'COVID-19': ('2020-02-19', '2020-03-23'), 'Terra/Luna': ('2022-05-05', '2022-05-12'), 'FTX': ('2022-11-05', '2022-11-21')}
HILL_FRAC = 0.025
VR_Q = (2, 5, 10)
VR_WINDOW, VR_STEP = 500, 21
SUBPERIODS = [('2014-09-17', '2017-12-31'), ('2018-01-01', '2021-12-31'), ('2022-01-01', '2026-09-18')]
CORR_WINDOW = 250
ALPHA_VAR, ALPHA_ES = 0.01, 0.025
POSITION = 100000
BT_START = '2018-01-01'
BT_WINDOW = 365
PEG_START = '2021-01-01'
SVB = ('2023-03-08', '2023-03-20')
LLAMA = 'https://stablecoins.llama.fi/stablecoincharts/all'
STABLE_ID = {'usdt': 1, 'usdc': 2, 'ust': 3, 'dai': 5}
DAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
WD_PERIODS = {'2014-2019': ('2014-09-17', '2019-12-31'), '2020-2023': ('2020-01-01', '2023-12-31'), '2024-2026': ('2024-01-01', '2026-09-18')}
SEED = 2026
_LLAMA = {}
TABLE_DIR = '.'
BINS = [0, 1, 5, 10, 50, 1000000000.0]
BIN_LAB = ['< 1 bp', '1-5 bp', '5-10 bp', '10-50 bp', '> 50 bp']


def register_extra():
    """Add USDC, DAI and the IBIT ETF to the list of named series of the course data loader."""
    for k, v in EXTRA.items():
        SERIES.setdefault(k, v)


def close(k, start=None, end=END):
    """Daily close (adjusted close for the ETF) on the series' own calendar (crypto: 7 days a week)."""
    register_extra()
    return load_close(k, start or START.get(k), end)


def returns(k, start=None, end=END):
    """Daily log returns in % on the series' own calendar."""
    register_extra()
    return log_returns(k, start or START.get(k), end)


def joint_returns(a, b, start=None, end=END):
    """Log returns in % of two series on their common days: prices are joined first, then differenced."""
    p = pd.concat([close(a, start, end), close(b, start, end)], axis=1).dropna()
    return (100 * np.log(p).diff()).dropna()


def hac_ols(y, X, lags=10):
    """OLS with Newey-West (HAC) standard errors."""
    return sm.OLS(y, sm.add_constant(X)).fit(cov_type='HAC', cov_kwds={'maxlags': lags})


def hill(x, k):
    """Hill (1975): alpha_hat(k) = [ (1/k) sum_{i=1..k} ln(X_(i) / X_(k+1)) ]^(-1), X_(1) >= X_(2) >= ... (Chapter 5)."""
    x = np.sort(np.asarray(x, float))[::-1]
    return 1.0 / np.mean(np.log(x[:k] / x[k]))


def describe(k, start=None, end=END):
    """Descriptive statistics of daily log returns in %, with annualisation at the actual frequency of the series;
    Hill tail indices of the losses (left tail) and of the gains (right tail), k = 2.5% of n."""
    r = returns(k, start, end)
    x = r.values
    n = len(x)
    ppy = periods_per_year(r)
    kk = int(HILL_FRAC * n)
    al, ar = hill(-x[x < 0], kk), hill(x[x > 0], kk)
    return {'n': n, 'first': r.index[0].date().isoformat(), 'ppy': float(ppy), 'mean': float(x.mean()),
            'sd': float(x.std(ddof=1)), 'skew': float(stats.skew(x)), 'exkurt': float(stats.kurtosis(x)),
            'min': float(x.min()), 'min_date': r.idxmin().date().isoformat(), 'max': float(x.max()),
            'max_date': r.idxmax().date().isoformat(), 'ann_mean': float(ppy * x.mean()),
            'ann_vol': float(np.sqrt(ppy) * x.std(ddof=1)), 'ann_vol_252': float(np.sqrt(252) * x.std(ddof=1)),
            'ann_mean_252': float(252 * x.mean()), 'share5': float(100 * np.mean(np.abs(x) > 5)),
            'k': kk, 'hill_left': float(al), 'hill_right': float(ar),
            'hill_left_lo': float(al * (1 - 1.96 / np.sqrt(kk))), 'hill_left_hi': float(al * (1 + 1.96 / np.sqrt(kk))),
            'hill_right_lo': float(ar * (1 - 1.96 / np.sqrt(kk))), 'hill_right_hi': float(ar * (1 + 1.96 / np.sqrt(kk))),
            'jb': float(stats.jarque_bera(x)[0])}


def acf(x, nlags):
    """Sample autocorrelations rho_hat(1..nlags)."""
    e = np.asarray(x, float) - np.mean(x)
    s = np.sum(e ** 2)
    return np.array([np.sum(e[k:] * e[:-k]) / s for k in range(1, nlags + 1)])


def ljung_box(x, m=10):
    """Ljung-Box Q(m) = T(T+2) sum rho_k^2/(T-k) against chi-square(m)."""
    x = np.asarray(x, float)
    T = len(x)
    rho = acf(x, m)
    q = T * (T + 2) * np.sum(rho ** 2 / (T - np.arange(1, m + 1)))
    return {'q': float(q), 'p': float(stats.chi2.sf(q, m))}


def garch_fit(r, o=0):
    """GARCH(1,1) (o=0) or GJR-GARCH(1,1) (o=1) with constant mean and standardised Student-t innovations."""
    am = arch_model(r, mean='Constant', vol='GARCH', p=1, o=o, q=1, dist='t')
    return am.fit(disp='off', options={'maxiter': 2000})


def hs_var(x, a=ALPHA_VAR):
    """Historical-simulation VaR_a = -r_(k), k = ceil(n a) (a positive loss, in %)."""
    s = np.sort(np.asarray(x, float))
    return float(-s[int(np.ceil(len(s) * a)) - 1])


def hs_es(x, a=ALPHA_ES):
    """Historical-simulation ES_a = minus the mean of the k = ceil(n a) smallest returns."""
    s = np.sort(np.asarray(x, float))
    return float(-s[:int(np.ceil(len(s) * a))].mean())


def std_t_q(nu, a):
    """a-quantile of the Student-t scaled to unit variance."""
    return float(stats.t.ppf(a, nu) * np.sqrt((nu - 2) / nu))


def std_t_es(nu, a):
    """ES_a of the unit-variance Student-t (a positive number)."""
    q = stats.t.ppf(a, nu)
    return float(np.sqrt((nu - 2) / nu) * stats.t.pdf(q, nu) / a * (nu + q ** 2) / (nu - 1))


def kupiec(x, n, a=ALPHA_VAR):
    """Kupiec (1995) proportion-of-failures LR test: x exceptions in n days."""
    pi = x / n
    l0 = (n - x) * np.log(1 - a) + x * np.log(a)
    l1 = (n - x) * np.log(1 - pi) + (x * np.log(pi) if x > 0 else 0.0)
    lr = -2 * (l0 - l1)
    return float(lr), float(stats.chi2.sf(lr, 1))


def peg_stats(k, start=PEG_START, end=END):
    """Peg deviation in basis points, d_t = 10 000 (P_t - 1), on daily closes and daily lows; AR(1) coefficient of
    the closing deviation and the half-life ln(0.5)/ln(phi) of a deviation."""
    register_extra()
    d = read_market(SERIES[k][0]).loc[start:end]
    dev = 1e4 * (d['close'] - 1)
    low = 1e4 * (d['low'] - 1)
    x = dev.values
    phi = np.corrcoef(x[1:], x[:-1])[0, 1]
    return {'n': int(len(dev)), 'mean': float(dev.mean()), 'sd': float(dev.std()), 'mad': float(np.median(np.abs(dev))),
            'gt10': float(100 * np.mean(np.abs(dev) > 10)), 'gt50': float(100 * np.mean(np.abs(dev) > 50)),
            'gt100': float(100 * np.mean(np.abs(dev) > 100)), 'close_min': float(dev.min()),
            'close_min_date': dev.idxmin().date().isoformat(), 'close_max': float(dev.max()),
            'close_max_date': dev.idxmax().date().isoformat(), 'low_min': float(low.min()),
            'low_min_date': low.idxmin().date().isoformat(), 'phi': float(phi),
            'half_life': float(np.log(0.5) / np.log(abs(phi)))}


def backtest_btc(start=BT_START, window=BT_WINDOW, k='btc'):
    """One-day VaR 1% forecasts for Bitcoin since `start`: historical simulation on the last 365 days and
    GARCH(1,1)-t re-estimated every 365 days on all earlier data; exceptions, rate, Kupiec test, exceptions by year
    (another crypto asset with k='eth')."""
    r = returns(k)
    idx = np.where(r.index >= pd.Timestamp(start))[0]
    hs = pd.Series([hs_var(r.values[i - window:i]) for i in idx], index=r.index[idx])
    g = pd.Series(index=r.index[idx], dtype=float)
    for j0 in range(0, len(idx), window):
        i0 = idx[j0]
        am = arch_model(r, mean='Constant', vol='GARCH', p=1, q=1, dist='t')
        res = am.fit(disp='off', last_obs=r.index[i0], options={'maxiter': 2000})
        f = res.forecast(horizon=1, start=r.index[i0 - 1], reindex=False)
        p = res.params
        q = std_t_q(p['nu'], ALPHA_VAR)
        sig = np.sqrt(f.variance.values[:, 0])
        # forecast made on day i-1 for day i
        fc = pd.Series(-(p['mu'] + sig * q), index=r.index[i0 - 1:])[:-1]
        fc.index = r.index[i0:]
        sl = r.index[idx[j0:j0 + window]]
        g.loc[sl] = fc.loc[sl].values
    out = {'start': r.index[idx[0]].date().isoformat(), 'n': int(len(idx))}
    df = pd.DataFrame({'r': r.iloc[idx], 'HS': hs, 'GARCH-t': g})
    for m in ('HS', 'GARCH-t'):
        hit = df['r'] < -df[m]
        x = int(hit.sum())
        lr, pv = kupiec(x, len(df))
        out[m] = {'x': x, 'rate': float(100 * x / len(df)), 'exp': float(0.01 * len(df)), 'lr': lr, 'p': pv,
                  'year': {str(y): int(v) for y, v in hit.groupby(df.index.year).sum().items()},
                  'mean_var': float(df[m].mean())}
    return out, df


def crisis_table(episodes=EPISODES):
    """Cumulative log return (in %) from the close of the first day to the close of the last day of each episode;
    Bitcoin's worst day in the episode."""
    out = {}
    for lab, (a, b) in episodes.items():
        row = {}
        for k in ('btc', 'eth', 'sp500', 'gold'):
            p = close(k, '2014-01-01')
            p0 = p.loc[:a].iloc[-1]
            p1 = p.loc[:b].iloc[-1]
            row[k] = float(100 * np.log(p1 / p0))
        r = returns('btc', a, b)
        row['btc_worst'] = float(r.min())
        row['btc_worst_date'] = r.idxmin().date().isoformat()
        row['start'], row['end'] = a, b
        out[lab] = row
    return out


def worst_days(q=0.05):
    """Mean returns of Bitcoin and gold on the S&P 500's worst 5% of days (prices joined on common days) and their
    correlation with the S&P 500 on those days (Baur and Lucey, 2010: hedge and safe haven)."""
    out = {}
    for k in ('btc', 'gold'):
        R = joint_returns(k, 'sp500')
        thr = R['sp500'].quantile(q)
        bad = R[R['sp500'] <= thr]
        t = stats.ttest_1samp(bad[k], 0.0)
        out[k] = {'n': int(len(R)), 'n_bad': int(len(bad)), 'thr': float(thr), 'mean_sp': float(bad['sp500'].mean()),
                  'mean': float(bad[k].mean()), 't': float(t.statistic), 'p': float(t.pvalue),
                  'mean_all': float(R[k].mean()), 'corr_bad': float(bad[k].corr(bad['sp500'])),
                  'share_neg': float(100 * np.mean(bad[k] < 0))}
    return out


def annualise(mu, sd, ppy_true=365, ppy_wrong=252):
    """Annual mean and volatility of daily returns (in %) with the true frequency and with a wrong one; the Sharpe
    ratio (risk-free rate 0) computed consistently and with mixed frequencies."""
    return {'mean_true': ppy_true * mu, 'mean_wrong': ppy_wrong * mu, 'vol_true': float(np.sqrt(ppy_true) * sd),
            'vol_wrong': float(np.sqrt(ppy_wrong) * sd), 'ratio': float(np.sqrt(ppy_true / ppy_wrong)),
            'under_vol': float(100 * (1 - np.sqrt(ppy_wrong / ppy_true))), 'under_mean': float(100 * (1 - ppy_wrong / ppy_true)),
            'sharpe': float(ppy_true * mu / (np.sqrt(ppy_true) * sd)),
            'sharpe_mixed': float(ppy_wrong * mu / (np.sqrt(ppy_true) * sd)),
            'sqrt_true': float(np.sqrt(ppy_true)), 'sqrt_wrong': float(np.sqrt(ppy_wrong))}


def drawdown_path(prices):
    """Running maximum, drawdown in % of each price, the maximum drawdown and the gain needed to recover from it."""
    p = np.asarray(prices, float)
    m = np.maximum.accumulate(p)
    dd = 100 * (p / m - 1)
    i = int(np.argmin(dd))
    peak = float(m[i])
    rec = next((j for j in range(i, len(p)) if p[j] >= peak), None)
    return {'max': m.tolist(), 'dd': dd.tolist(), 'mdd': float(dd[i]), 'trough': float(p[i]), 'peak': peak,
            'gain_needed': float(100 * (peak / p[i] - 1)), 'recovery_index': rec}


def peg_bp(k, days=('2023-03-10', '2023-03-11', '2023-03-12', '2023-03-13'), holding=1_000_000):
    """Deviation from 1 USD in basis points, 10 000 (P - 1), of the daily closes and of the lowest daily low;
    the loss of a holder who sells `holding` coins at the worst close and at the lowest low."""
    register_extra()
    d = read_market(SERIES[k][0])
    cl = {x: float(round(d.loc[x, 'close'], 4)) for x in days}
    low = float(round(d.loc[days[0]:days[-1], 'low'].min(), 4))
    worst = min(cl.values())
    return {'close': cl, 'bp': {x: 1e4 * (v - 1) for x, v in cl.items()}, 'low': low, 'low_bp': 1e4 * (low - 1),
            'loss_close': holding * (1 - worst), 'loss_low': holding * (1 - low), 'worst': worst}


def normal_crypto(W, mu, sd, h=10):
    """Normal VaR 1% and ES 2.5% of a position of value W (daily returns in %, simple-return approximation), and
    the h-day VaR 1% by the square-root-of-time rule (mean ignored)."""
    z1, z25 = stats.norm.ppf(ALPHA_VAR), stats.norm.ppf(ALPHA_ES)
    v1 = -(mu + sd * z1)
    e25 = -mu + sd * stats.norm.pdf(z25) / ALPHA_ES
    return {'var1': float(v1), 'es25': float(e25), 'var1_usd': W * v1 / 100, 'es25_usd': W * e25 / 100,
            'var_h': float(np.sqrt(h) * (-z1) * sd), 'var_h_usd': W * float(np.sqrt(h) * (-z1) * sd) / 100,
            'zs': float(-z1 * sd), 'fs': float(sd * stats.norm.pdf(z25) / ALPHA_ES)}


def worst_returns(k='btc', n=365, m=10, W=100_000):
    """The m smallest of the last n daily returns with their dates; historical VaR 1% (k = ceil(0.01 n)) and
    ES 2.5% (k = ceil(0.025 n)) in % and in USD for a position W (log returns: W (1 - exp(-VaR/100)))."""
    r = returns(k).iloc[-n:]
    s = r.sort_values().iloc[:m]
    k1, k25 = int(np.ceil(n * ALPHA_VAR)), int(np.ceil(n * ALPHA_ES))
    w = [float(round(x, 2)) for x in s.values]
    var1 = -w[k1 - 1]
    es25 = -float(np.mean(w[:k25]))
    return {'first': r.index[0].date().isoformat(), 'last': r.index[-1].date().isoformat(), 'worst': w,
            'dates': [d.date().isoformat() for d in s.index], 'k1': k1, 'k25': k25, 'var1': var1, 'es25': es25,
            'sum': -float(np.sum(w[:k25])), 'var1_usd': float(W * (1 - np.exp(-var1 / 100))),
            'es25_usd': float(W * (1 - np.exp(-es25 / 100))), 'e_var1': float(np.exp(-var1 / 100)),
            'e_es25': float(np.exp(-es25 / 100))}


def hill_plot(k, kmin=20, frac=0.10):
    """Hill tail index of the losses for k = kmin, ..., frac n."""
    x = -returns(k).values
    x = x[x > 0]
    ks = np.arange(kmin, int(frac * len(x)))
    return ks, np.array([hill(x, kk) for kk in ks])


def fisher_diff(r1, n1, r2, n2):
    """Fisher z test of equal correlations in two independent samples."""
    z = (np.arctanh(r2) - np.arctanh(r1)) / np.sqrt(1 / (n1 - 3) + 1 / (n2 - 3))
    return float(z), float(2 * stats.norm.sf(abs(z)))


def var_position(k='btc', W=100_000):
    """VaR 1% and ES 2.5% of a position W: historical simulation, Normal, Student-t (maximum likelihood) on the whole
    sample, GARCH(1,1)-t for the next day; in % and in USD."""
    r = returns(k)
    x = r.values
    mu, sd = x.mean(), x.std(ddof=1)
    nu, loc, sc = stats.t.fit(x)
    tq = stats.t.ppf(ALPHA_ES, nu)
    res = garch_fit(r)
    p = res.params
    s1 = float(np.sqrt(res.forecast(horizon=1, reindex=False).variance.values[-1, 0]))
    d = {'HS': (hs_var(x), hs_es(x)),
         'Normal': (float(-(mu + sd * stats.norm.ppf(ALPHA_VAR))), float(-mu + sd * stats.norm.pdf(stats.norm.ppf(ALPHA_ES)) / ALPHA_ES)),
         'Student-t': (float(-(loc + sc * stats.t.ppf(ALPHA_VAR, nu))), float(-loc + sc * stats.t.pdf(tq, nu) / ALPHA_ES * (nu + tq ** 2) / (nu - 1))),
         'GARCH-t': (float(-(p['mu'] + s1 * std_t_q(p['nu'], ALPHA_VAR))), float(-p['mu'] + s1 * std_t_es(p['nu'], ALPHA_ES)))}
    # log returns: a loss of v% in log terms is W (1 - exp(-v/100)) in USD
    out = {m: {'var': v, 'es': e, 'var_usd': float(W * (1 - np.exp(-v / 100))), 'es_usd': float(W * (1 - np.exp(-e / 100)))}
           for m, (v, e) in d.items()}
    out.update({'n': int(len(x)), 'first': r.index[0].date().isoformat(), 'nu': float(nu), 'g_sigma': s1,
                'g_nu': float(p['nu']), 'mu': float(mu), 'sd': float(sd)})
    return out

# Part A: computations on paper

- Do each computation on paper first; then check it with the code.

## A1 [Solved]: annualising Bitcoin

**Context.** Bitcoin daily log returns: mean 0.12%, standard deviation 3.5%; an analyst uses 252 days.

1. Compute the annual mean and volatility with 365 days.
2. Compute them with 252 days and say by how much they are understated.
3. Compute the Sharpe ratio (risk-free rate 0) correctly and with the mean × 252 and the volatility × √365.

**Report:** six numbers and one sentence.

In [8]:
# Solution
annualise(0.12, 3.5)

{'mean_true': 43.8,
 'mean_wrong': 30.24,
 'vol_true': 66.8674061108998,
 'vol_wrong': 55.560777532356404,
 'ratio': 1.2035001862952488,
 'under_vol': 16.909028233862276,
 'under_mean': 30.958904109589035,
 'sharpe': 0.6550276516986103,
 'sharpe_mixed': 0.4522382691179447,
 'sqrt_true': 19.1049731745428,
 'sqrt_wrong': 15.874507866387544}

**Interpretation of the result.** The 252-day convention understates the volatility by 17% and the mean by 31%; mixing the conventions lowers the Sharpe ratio.

## A2 [Proposed]: annualising Ethereum

**Context.** Ethereum daily log returns: mean 0.20%, standard deviation 5.0%. Model: A1.

1. Compute the annual mean and volatility with 365 and with 252 days.
2. Compute the ratio of the two volatilities.
3. Compute the correct and the mixed Sharpe ratio.

**Report:** seven numbers and one sentence.

In [ ]:
# Your code here


## A3 [Solved]: the maximum drawdown of a price path

**Context.** Monthly closes in thousand USD: 20, 35, 69, 50, 30, 16, 25, 45, 73.

1. Write the running maximum and the drawdown of each month.
2. Give the maximum drawdown and the gain needed to recover from the trough.
3. Say in which month the asset recovers.

**Report:** two numbers, a list and one sentence.

In [10]:
# Solution
drawdown_path([20, 35, 69, 50, 30, 16, 25, 45, 73])

{'max': [20.0, 35.0, 69.0, 69.0, 69.0, 69.0, 69.0, 69.0, 73.0],
 'dd': [0.0,
  0.0,
  0.0,
  -27.536231884057973,
  -56.52173913043479,
  -76.81159420289855,
  -63.76811594202898,
  -34.78260869565217,
  0.0],
 'mdd': -76.81159420289855,
 'trough': 16.0,
 'peak': 69.0,
 'gain_needed': 331.25,
 'recovery_index': 8}

**Interpretation of the result.** A fall of 77% needs a gain of 331% to recover; the record is regained in month 9.

## A4 [Proposed]: a second price path

**Context.** Quarterly closes of Ethereum in thousand USD: 1.4, 4.8, 3.2, 0.9, 1.2, 2.5, 4.9. Model: A3.

1. Write the running maximum and the drawdowns.
2. Give the maximum drawdown and the gain needed to recover.
3. Say in which quarter the asset recovers.

**Report:** two numbers, a list and one sentence.

In [ ]:
# Your code here


## A5 [Solved]: USDC in March 2023

**Context.** USDC daily closes from 10 to 13 March 2023 and the lowest daily low.

1. Compute the peg deviation of each close in basis points.
2. Compute the deviation of the lowest low.
3. Compute the loss of a holder of 1,000,000 USDC who sold at the worst close, and at the lowest low.

**Report:** five deviations, two losses and one sentence.

In [12]:
# Solution
peg_bp('usdc')

{'close': {'2023-03-10': 0.9995,
  '2023-03-11': 0.9715,
  '2023-03-12': 0.9921,
  '2023-03-13': 0.9989},
 'bp': {'2023-03-10': -4.999999999999449,
  '2023-03-11': -284.9999999999997,
  '2023-03-12': -79.00000000000018,
  '2023-03-13': -10.999999999999899},
 'low': 0.8774,
 'low_bp': -1226.0000000000005,
 'loss_close': 28499.99999999997,
 'loss_low': 122600.00000000004,
 'worst': 0.9715}

**Interpretation of the result.** The worst close was 285 bp below the peg and the low more than 12% below; who held until Monday lost almost nothing.

## A6 [Proposed]: DAI in March 2023

**Context.** DAI daily closes from 10 to 13 March 2023 and the lowest daily low. Model: A5.

1. Compute the four closing deviations and the deviation of the low in bp.
2. Compute the loss on 1,000,000 DAI sold at the worst close and at the low.
3. Explain why DAI, a crypto-backed coin, followed USDC.

**Report:** five deviations, two losses and one sentence.

In [ ]:
# Your code here


## A7 [Solved]: VaR of a Bitcoin position

**Context.** A position of 100,000 USD in Bitcoin.

1. With Normal simple returns ($\mu = 0.10\%$, $\sigma = 3.5\%$), compute VaR 1%, ES 2.5% and the 10-day VaR 1% in % and in USD.
2. From the 10 smallest of the last 365 log returns, compute the historical VaR 1% and ES 2.5% in % and in USD.

**Report:** ten numbers and one sentence.

In [14]:
# Solution
print(normal_crypto(100_000, 0.10, 3.5))
worst_returns('btc')

{'var1': 8.042217559142943, 'es25': 8.082309772704946, 'var1_usd': np.float64(8042.217559142943), 'es25_usd': np.float64(8082.309772704945), 'var_h': 25.747952691508438, 'var_h_usd': 25747.95269150844, 'zs': 8.142217559142942, 'fs': 8.182309772704945}


{'first': '2025-09-19',
 'last': '2026-09-18',
 'worst': [-15.23,
  -7.23,
  -6.77,
  -6.69,
  -5.46,
  -5.43,
  -5.32,
  -4.76,
  -4.69,
  -4.62],
 'dates': ['2026-02-05',
  '2025-10-10',
  '2026-01-31',
  '2026-06-02',
  '2025-11-14',
  '2025-11-20',
  '2026-01-29',
  '2025-11-04',
  '2026-01-20',
  '2026-06-05'],
 'k1': 4,
 'k25': 10,
 'var1': 6.69,
 'es25': 6.62,
 'sum': 66.2,
 'var1_usd': 6471.12744669579,
 'es25_usd': 6405.63431598703,
 'e_var1': 0.9352887255330421,
 'e_es25': 0.9359436568401297}

**Interpretation of the result.** With 365 days, $k = 4$ for VaR 1% and $k = 10$ for ES 2.5%; ES 2.5% can be below VaR 1%, because ES is larger than VaR only at the same level.

## A8 [Proposed]: VaR of an Ethereum position

**Context.** A position of 100,000 USD in Ethereum. Model: A7.

1. With Normal simple returns ($\mu = 0.20\%$, $\sigma = 5.0\%$), compute VaR 1%, ES 2.5% and the 10-day VaR 1%.
2. From the 10 smallest of the last 365 log returns, compute the historical VaR 1% and ES 2.5%.

**Report:** ten numbers and one sentence.

In [ ]:
# Your code here


# Part B: real data, inference and interpretation

## B1 [Solved]: Bitcoin against the S&P 500

**Question.** Is Bitcoin's distribution different from that of the S&P 500 only in scale, or also in the shape of its tails?

1. Compute the mean, the standard deviation, the skewness and the excess kurtosis of each series.
2. Annualise the mean and the volatility with the actual frequency and with 252 days.
3. Compute the Hill index of the left and of the right tail with $k = 2.5\%$ of $n$ and its 95% interval.
4. Draw the Hill plot of the losses for both series.
5. Interpretation: are Bitcoin's tails heavier than those of the S&P 500?

**Report:** a table of 12 numbers, the chart and two sentences.

In [16]:
# Solution
def b1_tails(keys=('btc', 'sp500'), save=True):
    """Moments, annualisation with the actual frequency and with 252 days, Hill tail indices with 95% intervals;
    the Hill plot of the losses."""
    out = {k: describe(k) for k in keys}
    if save:
        fig, ax = plt.subplots(figsize=(10, 3.8))
        for k in keys:
            ks, a = hill_plot(k)
            ax.plot(ks, a, color=COLORS[k], lw=1.4, label=f'{NAME[k]}: Hill estimate of the left tail index')
            n = out[k]['n']
            ax.axvline(int(0.025 * n), color=COLORS[k], lw=0.9, ls='--', label=f'{NAME[k]}: k = 2.5% of n')
        ax.set_xlabel('k (number of largest losses used)')
        ax.set_ylabel('Tail index alpha')
        st.legend_outside_bottom(ax, ncol=2, y=-0.18)
        st.check_no_grey(fig)
        st.save_fig('ch14_sem_b1')
    return out


b1 = b1_tails(save=False)
pd.DataFrame(b1).T[['n', 'mean', 'sd', 'skew', 'exkurt', 'ann_vol', 'ann_vol_252', 'hill_left', 'hill_left_lo', 'hill_left_hi', 'hill_right']].round(3)

,n,mean,sd,skew,exkurt,ann_vol,ann_vol_252,hill_left,hill_left_lo,hill_left_hi,hill_right
btc,4384,0.118056,3.485192,-0.704892,11.95772,66.614901,55.325711,2.706288,2.198226,3.21435,3.36793
sp500,3017,0.044443,1.110079,-0.636894,15.776163,17.60155,17.621958,2.684395,2.076859,3.291931,2.801042


**Interpretation of the result.** The left-tail indices of Bitcoin and the S&P 500 are almost equal and their intervals overlap: Bitcoin is about four times more volatile, but its tail has the same shape.

## B2 [Proposed]: Ethereum and Solana

**Question.** Do the conclusions of B1 hold for two other large crypto assets? Model: B1.

1. Compute the four moments of each series.
2. Annualise the volatility with 365 and with 252 days.
3. Compute both Hill indices with their 95% intervals.
4. Interpretation: which of the assets of B1 and B2 has the heaviest left tail?

**Report:** one table and two sentences.

In [ ]:
# Your code here


## B3 [Solved]: Bitcoin and the S&P 500 in calm and crisis

**Question.** Did the correlation between Bitcoin and equities change after 2020, and what happened in the 2020 and 2022 crises?

1. Compute the correlation before 2020 and from 2020 on, and test their equality with the Fisher $z$ test.
2. Compute the correlation of daily and of weekly returns year by year and draw both.
3. Compute the cumulative returns of both series in the COVID-19, Terra/Luna and FTX episodes.
4. Compute Bitcoin's mean return on the worst 5% of S&P 500 days.
5. Interpretation: does Bitcoin diversify an equity portfolio when it is most needed?

**Report:** six numbers, the chart and two sentences.

In [18]:
# Solution
def b3_corr(a='btc', b='sp500', save=True):
    """Correlation of daily log returns (prices joined on common days) before and after 2020 with the Fisher z test;
    daily and weekly correlation by year; cumulative returns in the 2020 and 2022 crisis episodes; mean return of
    `a` on the worst 5% of days of `b`."""
    R = joint_returns(a, b)
    pre, post = R.loc[:'2019-12-31'], R.loc['2020-01-01':]
    rp, rq = pre[a].corr(pre[b]), post[a].corr(post[b])
    z, p = fisher_diff(rp, len(pre), rq, len(post))
    se = 1 / np.sqrt(len(post) - 3)
    ci = np.tanh(np.arctanh(rq) + np.array([-1.96, 1.96]) * se)
    yr = R.groupby(R.index.year).apply(lambda g: g[a].corr(g[b]))
    wk = pd.concat([close(a), close(b)], axis=1).dropna().resample('W-FRI').last()
    W = np.log(wk).diff().dropna()
    wyr = W.groupby(W.index.year).apply(lambda g: g[a].corr(g[b]))
    thr = R[b].quantile(0.05)
    bad = R[R[b] <= thr]
    out = {'n': int(len(R)), 'n_pre': int(len(pre)), 'n_post': int(len(post)), 'pre': float(rp), 'post': float(rq),
           'z': z, 'p': p, 'ci_lo': float(ci[0]), 'ci_hi': float(ci[1]),
           'year': {str(y): float(v) for y, v in yr.items()}, 'week_year': {str(y): float(v) for y, v in wyr.items()},
           'weekly_all': float(W[a].corr(W[b])), 'daily_all': float(R[a].corr(R[b])),
           'bad_mean': float(bad[a].mean()), 'bad_n': int(len(bad)), 'bad_mean_b': float(bad[b].mean()),
           'crisis': {lab: {k: v for k, v in d.items() if k in (a, b)} for lab, d in crisis_table().items()}}
    if save:
        fig, ax = plt.subplots(figsize=(10, 3.8))
        ys = [y for y in yr.index if str(y) in out['week_year']]
        x = np.arange(len(ys))
        ax.bar(x - 0.2, [yr[y] for y in ys], width=0.4, color=st.MainBlue, label='Daily returns')
        ax.bar(x + 0.2, [wyr[y] for y in ys], width=0.4, color=st.Orange, label='Weekly returns (Friday to Friday)')
        ax.axhline(0, color='black', lw=0.6)
        ax.set_xticks(x)
        ax.set_xticklabels([str(y) for y in ys], rotation=45)
        ax.set_ylabel(f'Correlation, {NAME[a]} and {NAME[b]}')
        st.legend_outside_bottom(ax, ncol=2, y=-0.22)
        st.check_no_grey(fig)
        st.save_fig('ch14_sem_b3')
    return out


b3 = b3_corr(save=False)
{k: b3[k] for k in ['pre', 'post', 'z', 'p', 'ci_lo', 'ci_hi', 'daily_all', 'weekly_all', 'bad_mean', 'crisis']}

{'pre': 0.017806688748930686,
 'post': 0.3892625720717569,
 'z': 10.709730141568514,
 'p': 9.162851462644632e-27,
 'ci_lo': 0.3480017366008848,
 'ci_hi': 0.42901828675178433,
 'daily_all': 0.23728459085855466,
 'weekly_all': 0.18607624650419222,
 'bad_mean': -2.7716111665604517,
 'crisis': {'COVID-19': {'btc': -40.63909377366025,
   'sp500': -41.437912052076975},
  'Terra/Luna': {'btc': -23.042777077099846, 'sp500': -5.36940495722215},
  'FTX': {'btc': -29.86893152894543, 'sp500': 4.647948419402705}}}

**Interpretation of the result.** The correlation rose from about 0 to about 0.4 after 2020 (Fisher $z$ above 10), and on the worst S&P 500 days Bitcoin falls as much as the index: no diversification when it is most needed.

## B4 [Proposed]: Bitcoin and gold, Ethereum and the S&P 500

**Question.** Is Bitcoin linked with gold, and does Ethereum behave like Bitcoin towards equities? Model: B3.

1. Compute both correlations before and after 2020 and test their equality.
2. Compute the correlations of 2022 and 2025, daily and weekly.
3. Compute the mean return of Bitcoin on the worst 5% of gold days, and of Ethereum on the worst 5% of S&P 500 days.
4. Interpretation: is Bitcoin "digital gold"?

**Report:** one table and two sentences.

In [ ]:
# Your code here


## B5 [Solved]: how stable is USDC?

**Question.** How far and for how long does USDC move away from 1 USD?

1. Compute the peg deviations in bp and their standard deviation and median absolute value.
2. Compute the share of days in the classes below 1, 1–5, 5–10, 10–50 and above 50 bp, and draw them.
3. Fit an AR(1) to the deviation and compute the half-life of a deviation.
4. Compare the standard deviation before 2023 and after April 2023.
5. Interpretation: can a treasurer treat USDC as cash?

**Report:** six numbers, the chart and two sentences.

In [20]:
# Solution
def b5_peg(k='usdc', save=True):
    """Peg statistics since 2021 (basis points), the distribution of the absolute closing deviation, the AR(1)
    coefficient with its standard error and the half-life of a deviation."""
    out = peg_stats(k)
    register_extra()
    d = read_market(SERIES[k][0]).loc['2021-01-01':END]
    dev = 1e4 * (d['close'] - 1)
    h = np.histogram(np.abs(dev), bins=BINS)[0]
    out['bins'] = {l: float(100 * c / len(dev)) for l, c in zip(BIN_LAB, h)}
    x = dev.values
    X = np.column_stack([np.ones(len(x) - 1), x[:-1]])
    b, *_ = np.linalg.lstsq(X, x[1:], rcond=None)
    e = x[1:] - X @ b
    se = np.sqrt(np.sum(e ** 2) / (len(e) - 2) / np.sum((x[:-1] - x[:-1].mean()) ** 2))
    out['ar'] = float(b[1])
    out['ar_se'] = float(se)
    out['ar_hl'] = float(np.log(0.5) / np.log(abs(b[1])))
    d23 = dev.loc['2023-01-01':]
    out['sd_2023'] = float(dev.loc['2021-01-01':'2022-12-31'].std())
    out['sd_after'] = float(dev.loc['2023-04-01':].std())
    out['mad_after'] = float(np.median(np.abs(dev.loc['2023-04-01':])))
    if save:
        fig, ax = plt.subplots(figsize=(10, 3.6))
        ax.bar(BIN_LAB, [out['bins'][l] for l in BIN_LAB], color=COLORS[k], label=f'{NAME[k]}: share of days (%)')
        for i, l in enumerate(BIN_LAB):
            ax.text(i, out['bins'][l] + 1, f"{out['bins'][l]:.1f}%", ha='center', color='black', fontsize=11)
        ax.set_xlabel('Absolute deviation of the daily close from 1 USD')
        ax.set_ylabel('% of days')
        st.legend_outside_bottom(ax, ncol=1, y=-0.25)
        st.check_no_grey(fig)
        st.save_fig('ch14_sem_b5')
    return out


b5 = b5_peg('usdc', save=False)
b5

{'n': 2087,
 'mean': -0.21024051269764416,
 'sd': 7.482593378656689,
 'mad': 1.1262549999990767,
 'gt10': 1.5333013895543843,
 'gt50': 0.1437470052707235,
 'gt100': 0.09583133684714902,
 'close_min': -284.9999999999997,
 'close_min_date': '2023-03-11',
 'close_max': 104.96209999999894,
 'close_max_date': '2021-04-17',
 'low_min': -1226.0000000000005,
 'low_min_date': '2023-03-11',
 'phi': 0.2901633104399123,
 'half_life': 0.5602043224793956,
 'bins': {'< 1 bp': 45.99904168663153,
  '1-5 bp': 48.538572113080974,
  '5-10 bp': 3.9290848107331096,
  '10-50 bp': 1.3895543842836608,
  '> 50 bp': 0.14374700527072354},
 'ar': 0.2901699059445455,
 'ar_se': 0.02096344143031375,
 'ar_hl': 0.5602146139165126,
 'sd_2023': 5.793846069981639,
 'sd_after': 1.5132727636790044,
 'mad_after': 0.9619149999995358}

**Interpretation of the result.** On a typical day USDC is within a few bp of the peg and a deviation halves within a day; but the tail is real: one weekend in March 2023 cost up to 3% at the close and 12% at the low.

## B6 [Proposed]: USDT and DAI

**Question.** Are USDT and DAI as stable as USDC? Model: B5.

1. Compute the standard deviation, the median absolute deviation and the shares beyond 10 and 50 bp.
2. Find the worst close and the worst low of each coin, with their dates.
3. Fit the AR(1) and compute the half-lives.
4. Interpretation: which of the three coins keeps the peg best in normal times, and which in a crisis?

**Report:** one table and two sentences.

In [ ]:
# Your code here


## B7 [Solved]: risk of a Bitcoin position

**Question.** How much can a position of 100,000 USD in Bitcoin lose in one day, and which model would have passed a backtest?

1. Compute VaR 1% and ES 2.5% by historical simulation, Normal, Student-t (maximum likelihood) and GARCH(1,1)-t for the next day.
2. Convert them into USD with $W(1 - e^{-v/100})$.
3. Backtest the one-day VaR 1% since 2018 (historical simulation on 365 days, GARCH(1,1)-t re-estimated every 365 days); count the exceptions and run the Kupiec test.
4. Interpretation: which number would you report to a risk committee?

**Report:** a table of eight numbers, the chart, two test results and two sentences.

In [22]:
# Solution
def b7_var(k='btc', start='2018-01-01', save=True):
    """VaR of a 100 000 USD position and a backtest of the one-day VaR 1% since `start` (historical simulation on
    365 days and GARCH(1,1)-t re-estimated every 365 days); the left tail with the four VaR 1% values."""
    out = {'var': var_position(k)}
    bt, df = backtest_btc(start=start, k=k)
    out['bt'] = bt
    if save:
        x = returns(k).values
        fig, ax = plt.subplots(figsize=(10, 3.8))
        ax.hist(x[x < -3], bins=60, color=st.MainBlue, alpha=0.75, label='Daily returns below -3%')
        cols = {'HS': st.Amber, 'Normal': st.Forest, 'Student-t': st.Purple, 'GARCH-t': st.IDAred}
        for m, c in cols.items():
            ax.axvline(-out['var'][m]['var'], color=c, lw=1.6, ls='--', label=f'minus VaR 1%: {m}')
        ax.set_xlim(-30, -3)
        ax.set_xlabel('Daily log return (%)')
        ax.set_ylabel('Number of days')
        st.legend_outside_bottom(ax, ncol=3, y=-0.2)
        st.check_no_grey(fig)
        st.save_fig('ch14_sem_b7')
    return out


b7 = b7_var('btc', save=False)
print(pd.DataFrame({m: b7['var'][m] for m in ['HS', 'Normal', 'Student-t', 'GARCH-t']}).round(2))
{m: {c: b7['bt'][m][c] for c in ['x', 'rate', 'exp', 'lr', 'p']} for m in ['HS', 'GARCH-t']}

               HS   Normal  Student-t  GARCH-t
var         10.54     7.99      11.06     7.37
es          10.84     8.03      13.37     8.11
var_usd  10006.13  7678.87   10474.44  7105.02
es_usd   10276.74  7715.72   12514.92  7785.94


{'HS': {'x': 29,
  'rate': 0.9110901665095822,
  'exp': 31.830000000000002,
  'lr': 0.2619629475898364,
  'p': 0.6087747795220853},
 'GARCH-t': {'x': 41,
  'rate': 1.2880929940307886,
  'exp': 31.830000000000002,
  'lr': 2.4460625833966674,
  'p': 0.1178200772868695}}

**Interpretation of the result.** Report the historical VaR 1% as the long-run number and the GARCH-t VaR as today's number; the Normal VaR is about a quarter too low; both backtested models pass the Kupiec test.

## B8 [Proposed]: risk of an Ethereum position

**Question.** Does the conclusion of B7 hold for Ethereum? Model: B7.

1. Compute VaR 1% and ES 2.5% by the four methods of B7, in % and in USD.
2. Backtest the two one-day VaR 1% forecasts since 2019 and run the Kupiec test.
3. Find the largest number of exceptions in one calendar year for each model.
4. Interpretation: which model would you keep for Ethereum?

**Report:** one table and two sentences.

In [ ]:
# Your code here


# Part C: open questions and AI critique

## C1 [Proposed]: did the spot ETFs change Bitcoin?

**Question.** Since 11 January 2024 US investors can hold Bitcoin through spot ETFs traded on weekdays; did Bitcoin's volatility, its weekend pattern or its link with equities change? Models: B1, B3.

1. Compute the annualised volatility two years before and two years after the launch, and test the equality of variances (Brown–Forsythe).
2. Compute the ratio of weekend to weekday variance in both windows.
3. Compute the correlation with the S&P 500 and the GARCH(1,1)-t persistence in both windows.
4. Interpretation: can a before–after comparison prove an effect of the ETFs?

**Report:** a table and a plan for a project.

In [ ]:
# Your code here


## C2 [Proposed]: audit an AI answer

**Context.** An AI assistant summarised the risks of Bitcoin and stablecoins. It answered:

- (a) Bitcoin's annual volatility is the daily standard deviation times √252;
- (b) the "VaR 99%" of 100,000 USD in Bitcoin is a negative number of dollars equal to 1,000 times the VaR in %;
- (c) on 11 March 2023 USDC lost 2.85 basis points;
- (d) TerraUSD was a fiat-backed stablecoin with US Treasury bills as reserves;
- (e) Bitcoin is a safe haven because its correlation with the S&P 500 is about zero;
- (f) Bitcoin returns are less volatile at weekends than on weekdays.

1. For each statement, say whether it is correct; if not, give the correct statement and the correct number.

**Report:** six verdicts with one line of justification each.

In [ ]:
# Your code here
